### ABC-SS While-Loop Variation

For each simulation level (except the initial one), the algorithm performs the following steps:

1. **Sort samples by error**  
   The simulations from the previous level are sorted in ascending order of their error.

2. **Define the acceptance threshold (ε)**  
   The threshold is computed as the `P0` percentile of the error distribution of the sorted population.  
   Only samples with error less than or equal to this threshold are accepted.

3. **Select seeds**  
   The best samples (those below the acceptance threshold) are selected as seeds.

4. **Generate new samples**  
   - Each seed is perturbed several times by adding Gaussian noise with a standard deviation that decreases across levels.  
   - Each new sample is evaluated via a forward pass through the network and its error is computed.  
   - If the error satisfies the acceptance criterion, the mutated sample is stored; otherwise, the previous seed is kept.

5. **Update the population**  
   The accepted seeds and the newly generated subsets are merged to form the population for the next level.

6. **Store level results**  
   - Best model at this level (prediction and minimum error).  
   - Percentage of samples accepted at this level.

### Role of `error_threshold` and stopping criteria

Instead of running a fixed number of levels, the while-loop stops according to **performance-based criteria**:

- `error_threshold` (e.g. `error_threshold = 0.2`) defines a **target maximum error** for the best model.  
  - At each iteration, `best_error` is updated as the minimum error in the current population.  
  - The loop condition `while best_error > error_threshold and sim_level < max_iter:` means:
    - **Continue simulating** as long as the best model is still worse than the desired error, and  
    - The number of levels has not exceeded `max_iter`.

In addition to this main condition, several **safety stopping rules** are applied inside the loop:

- If the **acceptance rate** at the current level becomes too low (e.g. `< 0.0009`), the loop stops because the sampler is barely finding acceptable samples.  
- If the **best error stagnates** (the last few `best_errors` are essentially identical), the loop stops because further levels are not improving the model.  
- If the **proposal standard deviation** `prop_std` becomes non-positive (numerical safeguard), the loop is also terminated.

Together, these conditions turn the ABC-SS procedure into an **adaptive simulation**, which stops automatically once the model is “good enough” (according to `error_threshold`) or when the algorithm detects that further progress is unlikely.

> Apart from this while-loop and its stopping logic, the remaining blocks of the notebook (network definition, data generation, metric, plotting, etc.) are identical to those in the original ABC-SS implementation.

In [ ]:
error_treshold = 0.2
sim_level = 0  # Current simulation level
best_error = Ranked_ABCSampAcc[0, -1]  # Best error from the prior population
min_std = 0.01  # Minimum proposed standard deviation
decay_rate = 0.1  # Decrease rate of proposed std
best_errors = []

while best_error > error_threshold and sim_level < max_iter:
    # Sort samples by error (ascending)
    SampSort = ABCSampAcc[ABCSampAcc[:, -1].argsort()]

    # Compute acceptance threshold ε (percentile P0 of error)
    per_ind = np.rint(P0 * N).astype(int)  # Index corresponding to P0 percentile
    intr_eps = SampSort[per_ind, -1]  # Maximum error accepted at this level
    acc_intr_eps[sim_level] = intr_eps

    # Proposed standard deviation (decreases with levels)
    prop_std = max(min_std, new_std - (sim_level + 1) * decay_rate)

    # Seeds = best samples below the P0 percentile
    seeds = SampSort[0:per_ind, :]

    # Empty array for new subsets generated in this level
    SubSetsABC = np.zeros(shape=(N - per_ind, nW + nb + 1))

    # Counters
    ll = 0  # Row index for SubSetsABC
    accept_m = 0  # Number of newly accepted samples

    # For each selected seed
    for k in range(0, per_ind):
        prev_W = seeds[k, 0:nW]    # Weights of the seed
        prev_b = seeds[k, nW : nW + nb]  # Bias of the seed
        prev_metric = seeds[k, -1]  # Error of the seed

        # Generate ~1/P0 new samples for each seed
        for j in range(1, np.rint(1 / P0).astype(int)):
            New_W = np.zeros(shape=(nW))
            New_b = np.zeros(shape=(nb))

            # Mutate weights and bias (add Gaussian noise)
            for i in range(nW):
                New_W[i] = prop_std * np.random.randn() + prev_W[i]
            for r in range(nb):
                New_b[r] = prop_std * np.random.randn() + prev_b[r]

            # Evaluate new sample
            MW = MatrixW(New_W)
            Mb = Matrixb(New_b)
            Ypred = fpass(X, MW, Mb)
            e = metric(Ypred, Y)

            if e <= intr_eps:  # If accepted
                accept_m += 1
                SubSetsABC[ll, 0:nW] = New_W
                SubSetsABC[ll, nW : nW + nb] = New_b
                SubSetsABC[ll, -1] = e
                prev_W, prev_b, prev_metric = New_W, New_b, e
            else:  # Otherwise, keep previous seed
                SubSetsABC[ll, 0:nW] = prev_W
                SubSetsABC[ll, nW : nW + nb] = prev_b
                SubSetsABC[ll, -1] = prev_metric

            ll += 1

    # Combine seeds and new subsets to form next population
    ABCSampAcc = np.concatenate((seeds, SubSetsABC), axis=0)
    ACC_Interm_SS[sim_level + 1, :] = ABCSampAcc

    # Save the best prediction at this level (for visualization)
    Ranked_ABCSampAcc = ABCSampAcc[ABCSampAcc[:, -1].argsort()]
    best_error = Ranked_ABCSampAcc[0, -1]  # Update best error
    best_errors.append(best_error)  # Save for stagnation analysis

    Wtemp = Ranked_ABCSampAcc[0, 0:nW]
    btemp = Ranked_ABCSampAcc[0, nW : nW + nb]
    MWtemp = MatrixW(Wtemp)
    Mbtemp = Matrixb(btemp)
    Ypred_temp = fpass(X, MWtemp, Mbtemp)
    YFinal.append(Ypred_temp)
    etemp = metric(Ypred_temp, Y)
    eFinal.append(etemp)

    # Calculate percentage of accepted samples at this level
    accept_persim.append(float(accept_m) / float(N))
    print(
        f"Level {sim_level + 1}, ε={intr_eps:.4f}, σ_prop={prop_std:.2f}, Accepted: ({accept_persim[-1] * 100:.5f}%)"
    )

    # Custom stopping criteria
    if accept_persim[-1] < 0.0009:
        print(" Stopped: acceptance rate too low.")
        break

    if sim_level > 3:
        recent = [round(e, 4) for e in best_errors[-4:]]
        if all(e == recent[0] for e in recent):
            print("Stopped: error stagnated in last 4 levels.")
            break

    if prop_std <= 0:
        print(
            " Stopped: proposal std ≤ 0."
        )  # Should never happen (lower bound is 0)
        break

    sim_level += 1

Nivel 1, ε=0.4554, σ_prop=0.65, Aceptados: (6.52400%)
Nivel 2, ε=0.2977, σ_prop=0.55, Aceptados: (1.64200%)
Nivel 3, ε=0.2347, σ_prop=0.45, Aceptados: (0.54400%)
Nivel 4, ε=0.2002, σ_prop=0.35, Aceptados: (0.30100%)
